<a href="https://colab.research.google.com/github/mafaiziyas/Wearable-AI-Barbell-Activity-Recognition-and-Rep-Counting-Engine/blob/main/scripts/Feature_Engineering_%26_PCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Feature Engineering

In [2]:
"""
Feature engineering for the Barbell Exercise Classifier.

Builds three types of features on top of the cleaned/resampled
accelerometer + gyroscope data, computed within each set so that
windows never bleed across exercise/rest boundaries or across sets:

1. Temporal (time-domain) features   -> rolling mean/std/min/max/median
2. Frequency (FFT-based) features    -> dominant freq (reps per second/Hz),
                                        freq-weighted avg (Weighted balance point across all frequencies present in the window/Hz),
                                        power spectral entropy (Smoothness vs. Messiness),
                                        max amplitude (How strong the main rep movement is/deg per s)
3. Cluster feature                    -> KMeans cluster id on the acc/gyr
                                         magnitude signals (fit on train only)

PCA is then fit on the engineered numeric features (train only) and used
to reduce dimensionality / de-correlate the feature set before modelling.
"""

'\nFeature engineering for the Barbell Exercise Classifier.\n \nBuilds three types of features on top of the cleaned/resampled\naccelerometer + gyroscope data, computed within each set so that\nwindows never bleed across exercise/rest boundaries or across sets:\n \n1. Temporal (time-domain) features   -> rolling mean/std/min/max/median\n2. Frequency (FFT-based) features    -> dominant freq (reps per second/Hz), \n                                        freq-weighted avg (Weighted balance point across all frequencies present in the window/Hz),\n                                        power spectral entropy (Smoothness vs. Messiness), \n                                        max amplitude (How strong the main rep movement is/deg per s)\n3. Cluster feature                    -> KMeans cluster id on the acc/gyr\n                                         magnitude signals (fit on train only)\n \nPCA is then fit on the engineered numeric features (train only) and used\nto reduce dimensionali

In [3]:
import numpy as np
import pandas as pd
from scipy.fft import fft, fftfreq
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [4]:
RAW_COLS = ["acc_x (g)", "acc_y (g)", "acc_z (g)",
            "gyr_x (deg/s)", "gyr_y (deg/s)", "gyr_z (deg/s)"]

SAMPLING_RATE_HZ = 10          # data resampled to 100ms -> 10 Hz
TEMPORAL_WINDOW = 10           # ~1s window for rolling stats
FREQ_WINDOW = 20               # ~2s window for FFT features

In [5]:
def add_magnitudes(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["acc_r"] = np.sqrt(df["acc_x (g)"]**2 + df["acc_y (g)"]**2 + df["acc_z (g)"]**2)
    df["gyr_r"] = np.sqrt(df["gyr_x (deg/s)"]**2 + df["gyr_y (deg/s)"]**2 + df["gyr_z (deg/s)"]**2)
    return df

In [6]:

def add_temporal_features(df: pd.DataFrame, window: int = TEMPORAL_WINDOW) -> pd.DataFrame:
    df = df.copy()
    cols = RAW_COLS + ["acc_r", "gyr_r"]
    for col in cols:
        g = df.groupby("set")[col]
        df[f"{col}_mean"] = g.rolling(window, min_periods=1).mean().reset_index(level=0, drop=True)
        df[f"{col}_std"] = g.rolling(window, min_periods=1).std().reset_index(level=0, drop=True)
        df[f"{col}_min"] = g.rolling(window, min_periods=1).min().reset_index(level=0, drop=True)
        df[f"{col}_max"] = g.rolling(window, min_periods=1).max().reset_index(level=0, drop=True)
        df[f"{col}_median"] = g.rolling(window, min_periods=1).median().reset_index(level=0, drop=True)
    # first-sample std is NaN (single point) -> fill with 0
    std_cols = [c for c in df.columns if c.endswith("_std")]
    df[std_cols] = df[std_cols].fillna(0.0)
    return df

In [7]:

def _fft_features_for_window(values: np.ndarray, fs: int = SAMPLING_RATE_HZ):
    n = len(values)
    if n < 2:
        return 0.0, 0.0, 0.0, 0.0
    vals = values - np.mean(values)
    freqs = fftfreq(n, d=1.0 / fs)
    amps = np.abs(fft(vals))
    pos = freqs > 0
    freqs, amps = freqs[pos], amps[pos]
    if len(amps) == 0 or amps.sum() == 0:
        return 0.0, 0.0, 0.0, 0.0
    max_amp = amps.max()
    dom_freq = freqs[np.argmax(amps)]
    weighted_avg_freq = float((freqs * amps).sum() / amps.sum())
    p = amps / amps.sum()
    p = p[p > 0]
    pse = float(-(p * np.log(p)).sum())
    return float(max_amp), float(dom_freq), weighted_avg_freq, pse

In [8]:
def add_frequency_features(df: pd.DataFrame, window: int = FREQ_WINDOW) -> pd.DataFrame:
    df = df.copy()
    cols = ["acc_r", "gyr_r"]
    out = {f"{c}_{stat}": np.zeros(len(df))
           for c in cols for stat in ["fft_maxamp", "fft_domfreq", "fft_wavg", "fft_pse"]}

    for set_id, idx in df.groupby("set").groups.items():
        idx = list(idx)
        sub = df.loc[idx]
        n = len(sub)
        for col in cols:
            series = sub[col].values
            for i in range(n):
                lo = max(0, i - window + 1)
                window_vals = series[lo:i + 1]
                max_amp, dom_freq, wavg, pse = _fft_features_for_window(window_vals)
                row_idx = idx[i]
                out[f"{col}_fft_maxamp"][df.index.get_loc(row_idx)] = max_amp
                out[f"{col}_fft_domfreq"][df.index.get_loc(row_idx)] = dom_freq
                out[f"{col}_fft_wavg"][df.index.get_loc(row_idx)] = wavg
                out[f"{col}_fft_pse"][df.index.get_loc(row_idx)] = pse

    for k, v in out.items():
        df[k] = v
    return df

In [9]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = add_magnitudes(df)
    df = add_temporal_features(df)
    df = add_frequency_features(df)
    return df



In [10]:

def get_feature_cols(df: pd.DataFrame):
    exclude = {"epoch (ms)", "participant", "label", "category", "set"} | set(RAW_COLS)
    return [c for c in df.columns if c not in exclude]

In [11]:
def fit_cluster_feature(train_df, feature_cols, k=6, random_state=42):
    cluster_input_cols = [c for c in feature_cols if c.startswith(("acc_r", "gyr_r"))]
    scaler = StandardScaler().fit(train_df[cluster_input_cols])
    km = KMeans(n_clusters=k, n_init=10, random_state=random_state)
    km.fit(scaler.transform(train_df[cluster_input_cols]))
    return scaler, km, cluster_input_cols




In [12]:
def apply_cluster_feature(df, scaler, km, cluster_input_cols):
    df = df.copy()
    df["cluster"] = km.predict(scaler.transform(df[cluster_input_cols]))
    dummies = pd.get_dummies(df["cluster"], prefix="cluster")
    return pd.concat([df, dummies], axis=1)

In [13]:
def fit_pca(train_X: pd.DataFrame, variance_threshold=0.90, random_state=42):
    scaler = StandardScaler().fit(train_X)
    pca_full = PCA(random_state=random_state).fit(scaler.transform(train_X))
    cumvar = np.cumsum(pca_full.explained_variance_ratio_)
    n_comp = int(np.searchsorted(cumvar, variance_threshold) + 1)
    n_comp = min(n_comp, train_X.shape[1])
    pca = PCA(n_components=n_comp, random_state=random_state).fit(scaler.transform(train_X))
    return scaler, pca, n_comp

In [14]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [15]:
import json
import warnings
import numpy as np
import pandas as pd
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Removed: from features import engineer_features, get_feature_cols, fit_cluster_feature, apply_cluster_feature, fit_pca

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

print("Loading data...")
train_data = pd.read_csv("/content/drive/MyDrive/Fitness Tracker/Intermediate Data Files (post Data_preprocessing)/train.csv")
val = pd.read_csv("/content/drive/MyDrive/Fitness Tracker/Intermediate Data Files (post Data_preprocessing)/val.csv")
test = pd.read_csv("/content/drive/MyDrive/Fitness Tracker/Intermediate Data Files (post Data_preprocessing)/test.csv")

print("Engineering temporal + frequency features...")
train_fe = engineer_features(train_data)
val_fe = engineer_features(val)
test_fe = engineer_features(test)

feat_cols = get_feature_cols(train_fe)

print("Fitting cluster feature on train...")
scaler_c, km, cluster_cols = fit_cluster_feature(train_fe, feat_cols, k=6)
train_fe = apply_cluster_feature(train_fe, scaler_c, km, cluster_cols)
val_fe = apply_cluster_feature(val_fe, scaler_c, km, cluster_cols)
test_fe = apply_cluster_feature(test_fe, scaler_c, km, cluster_cols)

cluster_dummy_cols = [c for c in train_fe.columns if c.startswith("cluster_")]
# make sure val/test have same dummy columns (in case a cluster id is missing)
for c in cluster_dummy_cols:
    for d in (val_fe, test_fe):
        if c not in d.columns:
            d[c] = 0
full_feat_cols = feat_cols + cluster_dummy_cols

print("Fitting PCA on train features (90% variance target)...")
scaler_p, pca, n_comp = fit_pca(train_fe[full_feat_cols], variance_threshold=0.90)
print(f"  -> {n_comp} PCA components selected from {len(full_feat_cols)} engineered features")

def to_pca(df):
    return pca.transform(scaler_p.transform(df[full_feat_cols]))

X_train = to_pca(train_fe)
X_val = to_pca(val_fe)
X_test = to_pca(test_fe)
le = LabelEncoder().fit(train_fe["label"])
y_train = le.transform(train_fe["label"])
y_val = le.transform(val_fe["label"])
y_test = le.transform(test_fe["label"])

print("Train shape:", X_train.shape, "Val shape:", X_val.shape, "Test shape:", X_test.shape)

models = {
    "NaiveBayes": GaussianNB(),
    "SVM": SVC(kernel="rbf", C=10, gamma="scale", random_state=RANDOM_STATE),
    "RandomForest": RandomForestClassifier(n_estimators=300, max_depth=None, random_state=RANDOM_STATE, n_jobs=-1),
    "NeuralNet": MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=2000, random_state=RANDOM_STATE, early_stopping=True),
}

results = {}
print("\n=== Training on TRAIN, evaluating on VAL (participant D) ===")
for name, model in models.items():
    model.fit(X_train, y_train)
    pred_val = model.predict(X_val)
    acc = accuracy_score(y_val, pred_val)
    f1 = f1_score(y_val, pred_val, average="macro")
    results[name] = {"val_acc": acc, "val_f1_macro": f1}
    print(f"{name:14s} val_acc={acc:.3f}  val_f1_macro={f1:.3f}")

print("\n=== Same trained models, evaluating on TEST (participant B, held out) ===")
for name, model in models.items():
    pred_test = model.predict(X_test)
    acc = accuracy_score(y_test, pred_test)
    f1 = f1_score(y_test, pred_test, average="macro")
    results[name]["test_acc"] = acc
    results[name]["test_f1_macro"] = f1
    print(f"{name:14s} test_acc={acc:.3f} test_f1_macro={f1:.3f}")
    results[name]["test_report"] = classification_report(
        le.inverse_transform(y_test), le.inverse_transform(pred_test), output_dict=True, zero_division=0)
    results[name]["test_confusion_matrix"] = confusion_matrix(y_test, pred_test, labels=list(range(len(le.classes_)))).tolist()
    results[name]["confusion_labels"] = list(le.classes_)

with open("results_val_test.json", "w") as f:
    json.dump(results, f, indent=2, default=float)

# ---------------------------------------------------------------
# Supplementary: participant-grouped 5-fold CV on ALL data combined
# so every model gets evaluated on every class at least once, since
# val/test individually don't cover all 6 labels (participant quirk).
# ---------------------------------------------------------------
print("\n=== Supplementary: participant-grouped CV on combined data (all 6 classes) ===")
all_df = pd.concat([train_data, val, test], ignore_index=True)
all_fe = engineer_features(all_df)
all_fe = apply_cluster_feature(all_fe, scaler_c, km, cluster_cols)
for c in cluster_dummy_cols:
    if c not in all_fe.columns:
        all_fe[c] = 0
X_all = pca.transform(scaler_p.transform(all_fe[full_feat_cols]))
y_all = le.transform(all_fe["label"])
groups = all_fe["participant"].values

n_groups = len(np.unique(groups))
gkf = GroupKFold(n_splits=n_groups)  # leave-one-participant-out (5 participants)

cv_results = {}
for name, model in models.items():
    preds = cross_val_predict(model, X_all, y_all, groups=groups, cv=gkf, n_jobs=-1)
    acc = accuracy_score(y_all, preds)
    f1 = f1_score(y_all, preds, average="macro")
    cv_results[name] = {
        "loso_acc": acc,
        "loso_f1_macro": f1,
        "report": classification_report(le.inverse_transform(y_all), le.inverse_transform(preds), output_dict=True, zero_division=0),
    }
    print(f"{name:14s} LOSO_acc={acc:.3f}  LOSO_f1_macro={f1:.3f}")

with open("results_loso_cv.json", "w") as f:
    json.dump(cv_results, f, indent=2, default=float)

print("\nDone.")

Loading data...
Engineering temporal + frequency features...
Fitting cluster feature on train...
Fitting PCA on train features (90% variance target)...
  -> 16 PCA components selected from 56 engineered features
Train shape: (14145, 16) Val shape: (2092, 16) Test shape: (1675, 16)

=== Training on TRAIN, evaluating on VAL (participant D) ===
NaiveBayes     val_acc=0.855  val_f1_macro=0.453
SVM            val_acc=1.000  val_f1_macro=1.000
RandomForest   val_acc=1.000  val_f1_macro=1.000
NeuralNet      val_acc=1.000  val_f1_macro=1.000

=== Same trained models, evaluating on TEST (participant B, held out) ===
NaiveBayes     test_acc=0.744 test_f1_macro=0.536
SVM            test_acc=0.811 test_f1_macro=0.722
RandomForest   test_acc=0.781 test_f1_macro=0.533
NeuralNet      test_acc=0.832 test_f1_macro=0.594

=== Supplementary: participant-grouped CV on combined data (all 6 classes) ===
NaiveBayes     LOSO_acc=0.788  LOSO_f1_macro=0.788
SVM            LOSO_acc=0.957  LOSO_f1_macro=0.960
Ran

In [17]:
import json
import warnings
import numpy as np
import pandas as pd
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Removed: from features import engineer_features, get_feature_cols, fit_cluster_feature, apply_cluster_feature, fit_pca

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

print("Loading data...")
train_data = pd.read_csv("/content/drive/MyDrive/Fitness Tracker/Intermediate Data Files (post Data_preprocessing)/train.csv")
val = pd.read_csv("/content/drive/MyDrive/Fitness Tracker/Intermediate Data Files (post Data_preprocessing)/val.csv")
test = pd.read_csv("/content/drive/MyDrive/Fitness Tracker/Intermediate Data Files (post Data_preprocessing)/test.csv")

print("Engineering temporal + frequency features...")
train_fe = engineer_features(train_data)
val_fe = engineer_features(val)
test_fe = engineer_features(test)

feat_cols = get_feature_cols(train_fe)

print("Fitting cluster feature on train...")
scaler_c, km, cluster_cols = fit_cluster_feature(train_fe, feat_cols, k=6)
train_fe = apply_cluster_feature(train_fe, scaler_c, km, cluster_cols)
val_fe = apply_cluster_feature(val_fe, scaler_c, km, cluster_cols)
test_fe = apply_cluster_feature(test_fe, scaler_c, km, cluster_cols)

cluster_dummy_cols = [c for c in train_fe.columns if c.startswith("cluster_")]
# make sure val/test have same dummy columns (in case a cluster id is missing)
for c in cluster_dummy_cols:
    for d in (val_fe, test_fe):
        if c not in d.columns:
            d[c] = 0
full_feat_cols = feat_cols + cluster_dummy_cols

print("Fitting PCA on train features (90% variance target)...")
schler_p, pca, n_comp = fit_pca(train_fe[full_feat_cols], variance_threshold=0.90)
print(f"  -> {n_comp} PCA components selected from {len(full_feat_cols)} engineered features")

def to_pca(df):
    return pca.transform(scaler_p.transform(df[full_feat_cols]))

X_train = to_pca(train_fe)
X_val = to_pca(val_fe)
X_test = to_pca(test_fe)
le = LabelEncoder().fit(train_fe["label"])
y_train = le.transform(train_fe["label"])
y_val = le.transform(val_fe["label"])
y_test = le.transform(test_fe["label"])

print("Train shape:", X_train.shape, "Val shape:", X_val.shape, "Test shape:", X_test.shape)

models = {
    "NaiveBayes": GaussianNB(),
    "SVM": SVC(kernel="rbf", C=10, gamma="scale", random_state=RANDOM_STATE),
    "RandomForest": RandomForestClassifier(n_estimators=300, max_depth=None, random_state=RANDOM_STATE, n_jobs=-1),
    "NeuralNet": MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=2000, random_state=RANDOM_STATE, early_stopping=True),
}

def evaluate(model, X, y_true):
    pred = model.predict(X)
    return {
        "accuracy": accuracy_score(y_true, pred),
        "f1_macro": f1_score(y_true, pred, average="macro"),
        "f1_weighted": f1_score(y_true, pred, average="weighted"),
    }, pred


results = {}
print(f"\n{'Model':14s} {'Split':6s} {'Acc':>7s} {'MacroF1':>9s} {'WeightedF1':>11s}")
for name, model in models.items():
    model.fit(X_train, y_train)
    results[name] = {}

    for split_name, X_split, y_split in [
        ("train", X_train, y_train),
        ("val", X_val, y_val),
        ("test", X_test, y_test),
    ]:
        metrics, pred = evaluate(model, X_split, y_split)
        results[name][split_name] = metrics
        print(f"{name:14s} {split_name:6s} {metrics['accuracy']:7.3f} "
              f"{metrics['f1_macro']:9.3f} {metrics['f1_weighted']:11.3f}")

    # keep full detail for the test split (report + confusion matrix)
    _, pred_test = evaluate(model, X_test, y_test)
    results[name]["test"]["report"] = classification_report(
        le.inverse_transform(y_test), le.inverse_transform(pred_test),
        output_dict=True, zero_division=0)
    results[name]["test"]["confusion_matrix"] = confusion_matrix(
        y_test, pred_test, labels=list(range(len(le.classes_)))).tolist()
    results[name]["test"]["confusion_labels"] = list(le.classes_)
    print()

with open("results_val_test.json", "w") as f:
    json.dump(results, f, indent=2, default=float)

print("Done.")

Loading data...
Engineering temporal + frequency features...
Fitting cluster feature on train...
Fitting PCA on train features (90% variance target)...
  -> 16 PCA components selected from 56 engineered features
Train shape: (14145, 16) Val shape: (2092, 16) Test shape: (1675, 16)

Model          Split      Acc   MacroF1  WeightedF1
NaiveBayes     train    0.862     0.860       0.860
NaiveBayes     val      0.855     0.453       0.914
NaiveBayes     test     0.744     0.536       0.772

SVM            train    0.996     0.996       0.996
SVM            val      1.000     1.000       1.000
SVM            test     0.811     0.722       0.798

RandomForest   train    1.000     1.000       1.000
RandomForest   val      1.000     1.000       1.000
RandomForest   test     0.781     0.533       0.782

NeuralNet      train    0.999     0.999       0.999
NeuralNet      val      1.000     1.000       1.000
NeuralNet      test     0.832     0.594       0.833

Done.
